In [ ]:
from pyexpat import model

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Aggiunta delle librerie per il grafico
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Creazione di Dati sintetici per classificazione binaria
np.random.seed(42)

# Una volta generati i dati li suddividiamo in training e test set
# Generiamo due predittori X (feature)
X = np.random.rand(100, 2)*10

# Y è la classe (0 o 1), basata su una semplice soglia di separazione lineare
# Usiamo solo una dimensione (x[:.0]) per poter visualizzare chiaramente la curva logistica
X_single = X[:,0].reshape(-1, 1) # Usiamo solo la prima colonna come predittore
y = (X_single.flatten() > 5).astype(int)

# 2. Suddivisione in Set di Addestramento e test set
X_train, X_test, y_train, y_test = train_test_split(X_single, y, test_size=0.3, random_state=42)

# 3. Inizializzazione e Addestramento del modello
model  = LogisticRegression(solver='liblinear',random_state=42)
model.fit(X_train, y_train)

# 4. Previsioni e Valutazione
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy del modello: {accuracy:.2f}")

# 5. Interpretazione dei Coefficienti
print(f"Intercetta (beta_0): {model.intercept_[0]:.2f}")
print(f"Intercetta (beta_1): {model.coef_[0][0]:.2f}")

# 6. Visualizzazione della curva logistica

# Calcolo delle probabilità del modello su un intervallo continuo di valori X
X_range = np.linspace(X_single.min(), X_single.max(), 300).reshape(-1, 1)
y_proba = model.predict_proba(X_range)[:, 1] # Prende la probabilità della classe 1

# Disegno dello scatter plot dei dati reali
plt.figure(figsize=(10,6))
sns.scatterplot(x=X_single.flatten(), y=y, hue=y, palette={0: 'blue', 1: 'red'}, style=y, markers=['o', 'X'], s=100 )

# Disegno della curva di probabilità logistica
plt.plot(X_range, y_proba, color='green', linewidth=2, label='Curva Logistica $P(Y=1|X)$')

# Disegno della linea di soglia di decisione (0.5) e della linea X di separazione
plt.axhline(0.5, color='black', linestyle='--', alpha=0.7, label='Soglia di Decisione (0.5)')





Esempio Dataset Breast Cancer

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X,y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))



In [18]:

import pandas as pd
coeffs = pd.Series(model.coef_[0], index=load_breast_cancer().feature_names)
coeffs.sort_values(ascending=False).head(10)


mean radius                2.120338
worst radius               1.460705
texture error              0.924139
perimeter error            0.408961
mean texture               0.079500
radius error               0.025795
mean area                 -0.000552
fractal dimension error   -0.000597
smoothness error          -0.009060
worst area                -0.025250
dtype: float64

Esempio dimostrazione che: Logistic Regression = Rete neurale con un solo neurone

Logistic Regression con scikit-learn

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Logistic Regression -> Rete neurale corrispondente

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 1),
    nn.Sigmoid(),
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(torch.tensor(X_train, dtype=torch.float32))
    loss = criterion(outputs.squeeze(), torch.tensor(y_train, dtype=torch.float32))
    loss.backward()
    optimizer.step()